# LatentMind V6 — Colab Notebook

**Training-first pipeline**: synthesizes agentic traces → trains the policy brain → loads the agent → runs tests.

The brain is a trained 3-head MLP. It picks one action at a time — rag · sql · chart · email · template — and re-decides after each step, stopping when its *continue* score drops below the seuil.

Requires: T4 or L4 GPU runtime.

In [ ]:
# ─── GPU CLEANUP — run this any time before re-running the agent cells ───────
# Releases all model weights from VRAM so you can reload without OOM.
# Safe to run even on the very first pass (does nothing if nothing is loaded).

import gc, sys, torch

def cleanup(verbose: bool = True, _globals: dict | None = None) -> None:
    freed: list[str] = []

    # ── 1. delete notebook-level globals that hold model references ───────
    # The `agent` object wraps the LangGraph graph; its node closures call
    # get_slm() / get_polisher() / get_brain() — keeping tensors alive even
    # after we null the module-level singletons. Kill it first.
    _g = _globals or {}
    for name in ("agent", "slm"):
        if name in _g:
            del _g[name]
            freed.append(f"global:{name}")

    # ── 2. SLM — main model + 0.5B drafter + polisher ─────────────────────
    try:
        import v6.slm as _m
        slm_obj = _m._slm
        if slm_obj is not None:
            for tid in list(slm_obj._store.keys()):
                slm_obj.clear_thread(tid)
            if hasattr(slm_obj, "_draft") and slm_obj._draft is not None:
                del slm_obj._draft        # 0.5B drafter
                slm_obj._draft = None
                slm_obj._draft_tokenizer = None
            if hasattr(slm_obj, "model"):
                del slm_obj.model         # main 3B / 7B weights
            _m._slm = None
            freed.append("SLM (main + drafter)")
        pol_obj = _m._polisher
        if pol_obj is not None:
            if hasattr(pol_obj, "model"):
                del pol_obj.model
            _m._polisher = None
            freed.append("polisher")
    except Exception:
        pass

    # ── 3. BGE-M3 encoder + retriever ─────────────────────────────────────
    try:
        import v6.knowledge as _m
        enc_obj = _m._encoder
        if enc_obj is not None:
            if hasattr(enc_obj, "model"):
                del enc_obj.model
            _m._encoder = None
            freed.append("BGE-M3")
        _m._retriever = None
    except Exception:
        pass

    # ── 4. brain head + schema ─────────────────────────────────────────────
    try:
        import v6.brain as _m
        if _m._brain is not None:
            _m._brain = None
            freed.append("brain head")
    except Exception:
        pass
    try:
        import v6.schema as _m
        _m._schema = None
    except Exception:
        pass

    # ── 5. garbage-collect (multiple passes to break cycles) ──────────────
    for _ in range(3):
        gc.collect()

    # ── 6. release CUDA allocator cache ───────────────────────────────────
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    if verbose:
        if freed:
            print("Released:", ", ".join(freed))
        else:
            print("Nothing was loaded — nothing to release")
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1e9
            total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"VRAM: {alloc:.2f} GB / {total:.1f} GB after cleanup")
        print("Ready to re-run the load cell.")

cleanup(verbose=True, _globals=globals())


In [ ]:
# Core inference + RAG
!pip install 'transformers>=4.46.0' 'sentence-transformers>=3.0.0' 'accelerate>=0.27.0'
print("✓ transformers, sentence-transformers, accelerate")

# Graph + math
!pip install 'langgraph>=0.2.0' 'bitsandbytes>=0.43.0' scipy matplotlib
print("✓ langgraph, bitsandbytes, scipy, matplotlib")

# Utils
!pip install jinja2 pydantic pymysql mysql-connector-python
print("✓ jinja2, pydantic, mysql connectors")

print("\n✓ Setup complete! Ready to load agent.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, shutil

REPO_URL = 'https://github.com/Hamza09Hamza/Latent-Djezzy.git'
REPO_DIR = '/content/Latent-Djezzy'
BRANCH   = 'v6-clean'

os.chdir('/content')

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    # Repo already present — pull latest without nuking trained models
    print('Repo found — pulling latest from origin/v6-clean ...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
    print('✓ repo updated to latest origin/v6-clean')
else:
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
        print('removed stale directory')
    subprocess.run(
        ['git', 'clone', '--depth=1', '--branch', BRANCH, REPO_URL, REPO_DIR],
        check=True)
    print(f'✓ cloned {BRANCH} → {REPO_DIR}')

# Flush any cached v6 modules so the freshly pulled code is picked up
for mod in list(sys.modules.keys()):
    if 'v6' in mod:
        del sys.modules[mod]

os.chdir(REPO_DIR)

# Show the exact commit so you can confirm you're on the latest
commit = subprocess.check_output(
    ['git', 'log', '--oneline', '-1'], cwd=REPO_DIR).decode().strip()
print(f'HEAD: {commit}')

head_path = 'models/brain_head.pt'
if os.path.isfile(head_path):
    size_mb = os.path.getsize(head_path) / 1e6
    print(f'✓ Trained brain head found ({size_mb:.1f} MB) — will skip training')
else:
    print('! Trained brain head NOT found — Cells 6-7 will build it (~2 min on T4)')


In [ ]:
import shutil, os

# Database location — checks LatentDjezzy folder first, falls back to MyDrive root
LOCAL_DB   = "/content/interndb.sqlite"

possible_locations = [
    "/content/drive/MyDrive/LatentDjezzy/interndb.sqlite",
    "/content/drive/MyDrive/interndb.sqlite",
]

DRIVE_DB = None
for loc in possible_locations:
    if os.path.isfile(loc):
        DRIVE_DB = loc
        break

if not DRIVE_DB:
    print("⚠ Database not found in:")
    for loc in possible_locations:
        print(f"    {loc}")
    print("\nPlease ensure interndb.sqlite is in /MyDrive/LatentDjezzy/ or /MyDrive/")
else:
    if not os.path.isfile(LOCAL_DB):
        shutil.copy(DRIVE_DB, LOCAL_DB)
        print(f"✓ copied {os.path.getsize(LOCAL_DB):,} bytes → {LOCAL_DB}")
    else:
        print("✓ SQLite already present:", LOCAL_DB)

# Create output dirs on Drive so charts and emails are persisted
output_base = "/content/drive/MyDrive/LatentDjezzy/v6_output"
for d in [
    f"{output_base}/charts",
    f"{output_base}/emails",
    f"{output_base}/reports",
]:
    os.makedirs(d, exist_ok=True)
print(f"✓ Output dirs ready on Drive → {output_base}/")


In [ ]:
import os, warnings, logging

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

os.environ["V6_USE_SQLITE"]    = "1"
os.environ["V6_SQLITE_PATH"]   = "/content/interndb.sqlite"

# ── SQL generator model size ──────────────────────────────────────────────
# "3b"  → Qwen2.5-Coder-3B-Instruct     (~6.4 GB fp16)  — safe baseline
# "4b"  → Qwen3-4B-Instruct-2507        (~8.0 GB fp16)  ← current default
#          text-only, native Instruct, 262K ctx, no thinking mode.
#          drafter = Qwen3-0.6B (shared tokenizer, 2-4x speed)
# "7b"  → Qwen2.5-Coder-7B-Instruct     (~14 GB fp16)   — needs V6_4BIT=1
#
# Note: Qwen3.5-4B is the multimodal branch (vision+text) — model_type
# `qwen3_5` only ships in transformers from git main and there's no
# Instruct head. Stock Colab transformers can't load it. We use the
# pure-text Qwen3-4B-Instruct-2507 instead (transformers >= 4.51 OK).
os.environ["V6_SLM_SIZE"]       = "4b"

os.environ["V6_4BIT"]           = "0"   # "1" only needed for 7B on 16 GB T4
os.environ["V6_SPECULATIVE"]    = "1"   # drafter → 2-4x speed
os.environ["V6_CONSTRAINED_SQL"] = "0"

os.environ["V6_POLISHER_HUB_ID"] = "Qwen/Qwen2.5-1.5B-Instruct"
os.environ["V6_SLM_OVERRIDE"]   = ""
os.environ["V6_OUTPUT_DIR"]     = "/content/drive/MyDrive/LatentDjezzy/v6_output"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
device   = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
size  = os.environ.get("V6_SLM_SIZE", "4b")
quant = "4-bit" if os.environ.get("V6_4BIT") == "1" else "fp16"

_SLM_LABEL = {"3b": "Qwen2.5-Coder-3B", "4b": "Qwen3-4B-Instruct-2507", "7b": "Qwen2.5-Coder-7B"}
_SLM_GB    = {"3b": 6.4, "4b": 8.0, "7b": 7.0 if quant == "4-bit" else 14.0}
slm_gb     = _SLM_GB.get(size, 8.0)
draft_gb   = 1.2 if size in ("4b",) else 1.0
total_need = slm_gb + 3.0 + 2.3 + draft_gb + 1.0
headroom   = total_gb - total_need
status     = "✓ fits" if headroom >= 2.0 else "⚠ tight"

print(f"GPU:                {device}  ({total_gb:.1f} GB)")
print(f"SQL gen model:      {_SLM_LABEL.get(size, f'Qwen-{size}')} ({quant})")
print(f"Polisher (3-role):  {os.environ['V6_POLISHER_HUB_ID']}")
print(f"Speculative dec.:   {'✓ ON' if os.environ.get('V6_SPECULATIVE') == '1' else '✗ OFF'}")
print(f"Constrained SQL:    {'✓ ON' if os.environ.get('V6_CONSTRAINED_SQL') == '1' else '✗ OFF'}")
print(f"Output dir:         /content/drive/MyDrive/LatentDjezzy/v6_output")
print(f"\nVRAM budget: ~{total_need:.1f} GB needed / {total_gb:.1f} GB available  "
      f"({headroom:.1f} GB headroom  {status})")

In [ ]:
import os

# Set True to re-synthesize traces even if brain_head.pt already exists.
# Always set True after pulling new code that changed brain_data.py.
FORCE_RETRAIN = True

head_path = 'models/brain_head.pt'
if not FORCE_RETRAIN and os.path.isfile(head_path):
    print('Skipping trace synthesis (FORCE_RETRAIN=False and head already exists)')
else:
    # Synthesize agentic traces — the editable policy spec — for the brain.
    # Output: v6/data/brain_train.jsonl
    print('Synthesizing agentic traces...')
    !python3 -m v6.brain_data
    !echo "Rows in brain_train.jsonl:" && wc -l v6/data/brain_train.jsonl


In [ ]:
import os

# Set True to retrain even if brain_head.pt already exists.
FORCE_RETRAIN = True

head_path = 'models/brain_head.pt'
if not FORCE_RETRAIN and os.path.isfile(head_path):
    print('Brain head already trained; skipping (set FORCE_RETRAIN=True to redo)')
else:
    if os.path.isfile(head_path):
        os.remove(head_path)  # remove stale checkpoint before retraining
    # Train the 3-head MLP (intent · action · continue).
    # 200 epochs on the 8k-row dataset ~ 3-4 min on T4.
    # Output: models/brain_head.pt
    print('Training brain head (200 epochs)...')
    !python3 -m v6.train_brain --epochs 200

if os.path.isfile(head_path):
    print('\n✓ Brain ready:', head_path)
else:
    print('\n✗ Training did not produce', head_path, '— check the log above')


In [ ]:
import sys, os, torch
sys.path.insert(0, "/content/Latent-Djezzy")

# Free any previously loaded models before (re-)loading — prevents OOM on re-run
try:
    cleanup(verbose=False, _globals=globals())
except NameError:
    pass  # cleanup not defined yet (first run) — fine

from v6.graph import LatentMindV6
from v6.slm import get_slm, get_polisher
from v6.brain import get_brain

print("Loading BGE-M3 encoder + SLM (downloads on first run)…")
agent = LatentMindV6()
get_slm()    # force-load the main model so VRAM is allocated before queries
get_brain()  # force-load the trained brain head (fails fast if not trained)

print("\nLoading polisher (1.5B — natural-language refiner)…")
try:
    get_polisher()
    print("✓ Polisher ready")
except Exception as _e:
    print(f"  Polisher unavailable ({_e}) — raw answers will be shown instead")

if torch.cuda.is_available():
    alloc_gb = torch.cuda.memory_allocated() / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU: {alloc_gb:.1f} GB / {total_gb:.1f} GB  "
          f"(headroom ~{total_gb - alloc_gb:.1f} GB)")


In [ ]:
from v6.config import V6Config

print("\n" + "="*60)
print("Configuration Verification")
print("="*60)
print(f"SLM size:             {V6Config.SLM_SIZE}")
print(f"SLM model:            {V6Config.slm_id()}")
print(f"Polisher:             {V6Config.POLISHER_HUB_ID}")
print(f"4-bit quantization:   {'✓ Enabled' if V6Config.USE_4BIT else '✗ Disabled'}")
print(f"Constrained SQL:      {'✓ ON' if V6Config.USE_CONSTRAINED_SQL else '✗ OFF'}")
print(f"Brain head:           {V6Config.BRAIN_HEAD_PATH}")
print(f"Brain seuil:          {V6Config.BRAIN_SEUIL}  (continue ≥ this → keep going)")
print(f"Brain max steps:      {V6Config.BRAIN_MAX_STEPS}")
print("="*60 + "\n")
print("Ready to run queries. Try: ask('Hello, what can you do?')")

In [ ]:
import time, os, sys
from v6.state import initial_state
from IPython.display import display, Image, Markdown

_DIM, _RESET = "\033[2m", "\033[0m"

def _type(text, delay=0.005):
    for ch in text:
        sys.stdout.write(ch)
        sys.stdout.flush()
        time.sleep(delay)

def ask(question, thread="main"):
    # One turn: stream the brain's thinking, then the terminal response.
    # The terminal has 3 roles:
    #   Analyst  → SQL data rows → insightful paragraph (same language as query)
    #   Polisher → RAG / definition / greeting → natural rewrite
    #   Clarifier→ errors, missing info, or unknown KPI → helpful clarification
    config = {"configurable": {"thread_id": thread}, "recursion_limit": 60}
    state  = initial_state(question, thread)

    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    t0 = time.time()

    shown = 0
    final_answer  = ""
    chart_path    = ""
    document_path = ""
    email_draft   = None
    intent        = ""
    exec_ok       = False   # did SQL actually return rows?
    timings       = {}

    for event in agent.graph.stream(state, config=config, stream_mode="updates"):
        for node, data in event.items():
            if not data:
                continue
            thoughts = data.get("thoughts")
            if thoughts and len(thoughts) > shown:
                for th in thoughts[shown:]:
                    if th.get("kind") == "thinking":
                        sys.stdout.write(_DIM + "  💭 ")
                        _type(th["text"])
                        print(_RESET)
                shown = len(thoughts)
            if data.get("timings"):
                timings = data["timings"]
            if node == "brain":
                intent = data.get("intent", intent)
                act    = data.get("next_action", "")
                cont   = data.get("continue_score", 0.0)
                print(f"{_DIM}     → {act}  (continue {cont:.2f}){_RESET}")
            elif node == "sql":
                # Track whether SQL actually produced rows — used below to
                # pick the right terminal role (Analyst vs Clarifier).
                exec_ok = data.get("exec_ok", exec_ok)
            elif node == "chart":
                chart_path = data.get("chart_path", "") or chart_path
            elif node == "template":
                document_path = data.get("document_path", "") or document_path
            elif node == "email":
                email_draft = data.get("email_draft") or email_draft
            elif node == "communicator":
                final_answer = data.get("final_answer", "")

    elapsed = time.time() - t0

    # ── 3-role terminal ───────────────────────────────────────────────────
    print(f"\nAnswer ({elapsed:.1f}s):")
    from v6.slm import get_polisher

    _FAILURE = ("couldn't build", "failed to run", "no matching rows",
                "wasn't able to pull", "no data found")
    is_failure = any(p in (final_answer or "").lower() for p in _FAILURE)
    email_no_recipient = bool(email_draft and email_draft.get("status") != "draft")

    def _stream_role(text, role):
        try:
            for tok in get_polisher().stream(text, question, role=role):
                print(tok, end="", flush=True)
            print()
        except Exception:
            print(text)

    if is_failure:
        # Clarifier explains what went wrong (period out of range, bad query…)
        _stream_role(final_answer, "clarify")
    elif email_no_recipient:
        # Analyze data (if present) then ask for a recipient.
        data_part = (final_answer or "").split("📧")[0].strip()
        if data_part:
            _stream_role(data_part, "analyze")
        _stream_role(
            "An email was drafted but no recipient was identified in the request.",
            "clarify")
    elif intent in ("greeting", "meta"):
        _stream_role(final_answer, "polish")
    elif intent == "definition" and final_answer:
        _stream_role(final_answer, "polish")
    elif intent == "data" and final_answer:
        if not exec_ok:
            # SQL ran but produced no rows / NULL, OR the KPI wasn't found.
            # Use Clarifier so it explains what's missing, not the Analyst.
            _stream_role(final_answer, "clarify")
        elif email_draft and email_draft.get("status") == "draft":
            # Email drafted — analyze the data part first (always "analyze":
            # single-value or multi-row, the Analyst handles both), then show draft.
            data_part = (final_answer or "").split("📧")[0].strip()
            if data_part:
                _stream_role(data_part, "analyze")
        elif document_path:
            # Report generated — the document IS the analysis; just print the
            # confirmation message (e.g. "Report generated from prior result.")
            print(final_answer or "(no answer)")
        else:
            # Pure data OR chart — always "analyze".
            # The Polisher is for RAG/definitions/greetings, NOT for numbers.
            _stream_role(final_answer, "analyze")
    else:
        print(final_answer or "(no answer)")

    # ── display chart inline ──────────────────────────────────────────────
    if chart_path and os.path.isfile(chart_path):
        print()
        display(Image(chart_path))
    elif chart_path:
        print(f"  (chart path reported but file missing: {chart_path})")

    # ── report ────────────────────────────────────────────────────────────
    if document_path:
        print(f"  Report saved → {document_path}")

    # ── email draft + save to Drive ───────────────────────────────────────
    if email_draft:
        print()
        to = email_draft.get("to") or "(no recipient — please specify one)"
        display(Markdown(f"**Email draft**\n\n"
                         f"**To:** {email_draft.get('to_name', '?')} <{to}>  \n"
                         f"**Subject:** {email_draft.get('subject', '')}\n\n"
                         f"---\n\n{email_draft.get('body', '')}"))
        try:
            import datetime
            from v6.config import V6Config
            email_dir = os.path.join(V6Config.output_dir(), "emails")
            os.makedirs(email_dir, exist_ok=True)
            stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            email_path = os.path.join(email_dir, f"email_{stamp}.txt")
            with open(email_path, "w") as f:
                f.write(f"To: {email_draft.get('to_name', '?')} <{to}>\n")
                f.write(f"Subject: {email_draft.get('subject', '')}\n\n")
                f.write(email_draft.get("body", ""))
            print(f"  Email saved → {email_path}")
        except Exception as e:
            print(f"  (could not save email: {e})")

    # ── performance breakdown ─────────────────────────────────────────────
    import torch
    brain_ms = sum(v for k, v in timings.items() if k.startswith("brain"))
    sql_ms   = sum(v for k, v in timings.items() if k.startswith("sql"))
    rag_ms   = timings.get("rag_ms", 0.0)
    n_ticks  = sum(1 for k in timings if k.startswith("brain"))
    line = (f"⏱  {n_ticks} brain ticks {brain_ms:.0f}ms · "
            f"rag {rag_ms / 1000:.2f}s · sql {sql_ms / 1000:.2f}s · "
            f"total {elapsed:.1f}s")
    if torch.cuda.is_available():
        line += f" · VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB"
    print(f"{_DIM}{line}{_RESET}")

    # ── free the KV cache ─────────────────────────────────────────────────
    try:
        from v6.slm import get_slm
        get_slm().clear_thread(thread)
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print()


In [ ]:
# Greeting in Darija / French mix
ask("Salam ! C'est quoi exactement ce que tu sais faire ?")

In [ ]:
# Definition — no sql, French query
ask("C'est quoi exactement le taux de désabonnement ?")

In [ ]:
# fpa_profitability, hard wilaya name with accent, relative time, French
ask("Montre-moi la marge brute pour Oum El Bouaghi le trimestre dernier")

In [ ]:
# prepaid_kpi, ranking, different KPI (churn not recharge), no specific wilaya
ask("Which wilaya had the lowest postpaid churn rate in Q2 2025?")

In [ ]:
# Chart trend, two hard wilayas — tests multi-series line + accent handling
ask("Plot the net income trend for Sidi Bel Abbès and M'Sila over 2025")

In [ ]:
# opex_capex, different categories, first half of year
ask("What's the total OPEX for marketing and distribution in the first half of 2025?")

In [ ]:
# postpaid_kpi, hard wilaya with accent + diaeresis — tests entity resolver
# NOTE: thread="main" — cells below (follow-up, template) continue this conversation
ask("Combien d'abonnés postpaid ont migré à Aïn Témouchent le mois dernier ?")

In [ ]:
# Follow-up — very hard wilaya name (two words + accent), inherits postpaid + migration
ask("et pour Bordj Bou Arréridj ?")

In [ ]:
# Follow-up — template from last_rows (cross-turn, no SQL re-run), French
ask("Mets-le dans un rapport")

In [ ]:
# prepaid_kpi + email — ARPU breakdown, email to CEO (mike.brown@company.com)
# Note: "finance director" in the contacts IS the test user (boukader hamza),
# so we use "CEO" here to demonstrate recipient resolution to a different person.
ask("Email the prepaid ARPU breakdown by wilaya to the CEO")

In [ ]:
# Unanswerable — fake KPI with a real hard wilaya to make it tricky
ask("What is the network latency score for Ghardaïa ?")

In [ ]:
# Multi-capability — hard wilayas, churn trend chart + email to ops director
ask("Chart the postpaid churn trend for Khenchela and Tissemsilt over 2025 and email it to the operations manager")

In [ ]:
# Times agent.graph.invoke directly (no UI polish / typewriter) so the numbers
# are the real engine cost. The brain MLP is milliseconds — the SLM dominates.
import time, torch
from v6.state import initial_state
from v6.slm import get_slm

BENCH = [
    "Salam ! C'est quoi exactement ce que tu sais faire ?",
    "C'est quoi exactement le taux de désabonnement ?",
    "Montre-moi la marge brute pour Oum El Bouaghi le trimestre dernier",
    "Which wilaya had the lowest postpaid churn rate in Q2 2025?",
    "Plot the net income trend for Sidi Bel Abbès and M'Sila over 2025",
    "Email the prepaid ARPU breakdown by wilaya to the CEO",
]

hdr = f"{'query':<46}{'intent':<11}{'steps':>6}{'brain':>9}{'sql':>9}{'total':>9}"
print(hdr)
print("-" * len(hdr))
tot = 0.0
for q in BENCH:
    t0 = time.time()
    r = agent.graph.invoke(
        initial_state(q, "bench"),
        {"configurable": {"thread_id": "bench"}, "recursion_limit": 60})
    dt = time.time() - t0
    tot += dt
    tm = r.get("timings", {})
    brain_ms = sum(v for k, v in tm.items() if k.startswith("brain"))
    sql_ms   = sum(v for k, v in tm.items() if k.startswith("sql"))
    print(f"{q[:46]:<46}{r.get('intent',''):<11}"
          f"{r.get('brain_step',0):>6}{brain_ms:>7.0f}ms"
          f"{sql_ms/1000:>8.2f}s{dt:>8.2f}s")
    get_slm().clear_thread("bench")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("-" * len(hdr))
print(f"{'AVERAGE per query':<46}{'':<11}{'':>6}{'':>9}{'':>9}"
      f"{tot/len(BENCH):>8.2f}s")
if torch.cuda.is_available():
    a = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM: {a:.2f} GB used / {t:.1f} GB total  ({t-a:.1f} GB free)")
    print("Brain overhead is the 'brain' column — a few ms per query; the "
          "SLM router+sqlgen is the cost centre.")


In [ ]:
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated() / 1e9
    reserv = torch.cuda.memory_reserved()  / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  allocated: {alloc:.2f} GB")
    print(f"GPU  reserved:  {reserv:.2f} GB")
    print(f"GPU  total:     {total:.2f} GB")

In [ ]:
# Direct streaming — bypasses the graph, useful for quick SLM checks.
from v6.slm import get_slm

slm = get_slm()
messages = [
    {"role": "system", "content": "You are a helpful telecom analyst."},
    {"role": "user",   "content": "List the top 3 KPIs for a telecom operator."},
]
print("Streaming response:")
for token in slm.stream_generate(messages, max_new_tokens=256):
    print(token, end="", flush=True)
print()